In [ ]:
# ============================================================
# Overfitting / Underfitting Check
# 过拟合 / 欠拟合检查
#
# Motivation / 目的:
#   Reporting Test MSE alone is insufficient to evaluate model quality.
#   A lower Test MSE could result from genuine generalisation improvement,
#   or simply from the model overfitting the training data.
#   By comparing Train MSE and Test MSE, we can confirm that the
#   improvement in Model 2 is real and trustworthy.
#
#   仅汇报测试集MSE不足以评估模型质量。
#   更低的测试集MSE可能来自真实的泛化能力提升，
#   也可能只是模型过拟合训练数据的结果。
#   通过对比训练集MSE和测试集MSE，我们可以确认模型2的性能提升是真实可靠的。
#
# Why Train vs Test (not Train vs Val) / 为什么用训练集 vs 测试集（而非验证集）:
#   In the final training stage, the training and validation sets are
#   merged together to maximise the amount of data available for training.
#   This means no independent validation set remains.
#   The held-out test set therefore serves as the reference for
#   evaluating generalisation performance.
#
#   在最终训练阶段，训练集和验证集被合并以最大化可用训练数据。
#   因此不再有独立的验证集可用。
#   保留的测试集因此作为评估泛化性能的参考集。
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ── Diagnosis helper / 诊断辅助函数 ──────────────────────────
def diagnose_final(model_name, train_mse, test_mse, threshold_ratio=1.5):
    """
    Print an overfitting/underfitting diagnosis based on Test/Train MSE ratio.
    根据测试集与训练集MSE比值，打印过拟合/欠拟合诊断结果。

    Diagnosis rules / 诊断规则:
        ratio > 1.5                      → Overfitting
        比值 > 1.5                        → 过拟合：测试集误差远大于训练集
        ratio < 1.1                      → Low generalisation gap; possible underfitting if both errors remain high
        比值 < 1.1                        → 泛化差距较小；若两者均较高则可能欠拟合
        Otherwise                        → Good fit
        否则                              → 拟合良好
    """
    # Compute ratio of test MSE to train MSE
    # If train_mse is zero, set ratio to infinity to avoid division by zero
    # 计算测试集MSE与训练集MSE的比值
    # 若训练集MSE为零则设为无穷大，避免除零错误
    ratio = test_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE       : {train_mse:,.4f}")
    print(f"  Test  MSE       : {test_mse:,.4f}")
    print(f"  Test/Train Ratio: {ratio:.3f}")

    # Diagnosis is based on relative comparison between train and test MSE,
    # rather than absolute thresholds, which are dataset-dependent.
    # 诊断基于训练集和测试集MSE的相对比较，而非绝对阈值，
    # 因为绝对阈值依赖于具体数据集的MSE量级。
    if ratio > threshold_ratio:
        # Test MSE is much larger than Train MSE
        # → model memorised training data but fails to generalise → overfitting
        # 测试集MSE远大于训练集MSE
        # → 模型过度记忆训练数据，泛化能力差 → 过拟合
        print("  ⚠ Diagnosis : OVERFITTING  — test MSE is much larger than train MSE")
        print("  ⚠ 诊断结果  : 过拟合 — 测试集误差远大于训练集误差")
    elif ratio < 1.1:
        # Train and test MSE are similar (ratio < 1.1)
        # If both values remain relatively high, this may indicate underfitting
        # — the model lacks sufficient capacity to learn the underlying pattern.
        # No absolute threshold is used here as MSE scale is dataset-dependent.
        # 训练集与测试集MSE相近（比值 < 1.1）
        # 若两者均处于相对较高的水平，则可能存在欠拟合
        # — 模型容量不足，无法学习数据中的潜在规律。
        # 此处不使用绝对阈值，因为MSE的量级取决于具体数据集。
        print("  ~ Diagnosis : LOW GENERALISATION GAP — train and test errors are similar;")
        print("                if both remain relatively high this may indicate underfitting")
        print("  ~ 诊断结果  : 泛化差距较小 — 训练集与测试集误差接近；")
        print("                若两者均较高则可能说明模型容量不足")
    else:
        # Train and test MSE are close and both reasonably low
        # → model generalises well → good fit
        # 训练集与测试集误差接近且均处于合理水平
        # → 模型泛化能力良好 → 拟合良好
        print("  ✓ Diagnosis : GOOD FIT     — train and test MSE are close and low")
        print("  ✓ 诊断结果  : 拟合良好 — 训练集与测试集误差接近且处于合理水平")
    print(f"{'='*55}")


# ============================================================
# MODEL 1 — Final Baseline
# 模型1 — 最终基线模型
#
# baseline_final_model was trained on the merged train+val set
# using only the original features, with no neighbourhood features.
# Both baseline_train_mse and baseline_test_mse_final were already
# computed immediately after baseline_final_model was trained,
# so we reuse them directly here without any additional computation.
#
# baseline_final_model 在合并后的 train+val 集上训练，仅使用原始特征。
# baseline_train_mse 和 baseline_test_mse_final 在
# baseline_final_model 训练完成后已经立即计算好，直接复用。
# ============================================================

# Reuse already-computed values — no re-training needed
# 直接复用已计算好的值，无需重新训练
m1_train_mse = baseline_train_mse        # already computed above / 原代码已算好
m1_test_mse  = baseline_test_mse_final   # already computed above / 原代码已算好

diagnose_final("Model 1 — Final Baseline", m1_train_mse, m1_test_mse)


# ============================================================
# MODEL 2 — Final Enhanced
# 模型2 — 最终增强模型（含邻域统计特征）
#
# model_enh_final was trained on the merged train+val set
# with neighbourhood-based price statistics added as extra inputs.
# enhanced_test_mse_final was already computed immediately after
# model_enh_final was trained, so we reuse it directly.
#
# However, Train MSE for Model 2 was never computed in the original
# code, so we compute it once here using the already-fitted model.
# No re-training is required.
#
# model_enh_final 在合并后的 train+val 集上训练，包含邻域统计特征。
# enhanced_test_mse_final 已在训练后立即计算，直接复用。
# 但模型2的训练集MSE在原代码中从未计算过，
# 因此在此处使用已训练好的模型计算一次，无需重新训练。
# ============================================================

# Reuse already-computed test MSE — no re-training needed
# 直接复用已计算好的测试集MSE，无需重新训练
m2_test_mse  = enhanced_test_mse_final   # already computed above / 原代码已算好

# Train MSE for Model 2 was not computed in the original code
# Compute it once here using the already-fitted model_enh_final
# 模型2的训练集MSE在原代码中未计算，此处用已训练好的模型补充计算一次
m2_train_mse = mean_squared_error(       # not in original code, compute once here
    y_train_final,                        # 原代码未计算，在此计算一次
    model_enh_final.predict(X_train_enh_final)
)

diagnose_final("Model 2 — Final Enhanced", m2_train_mse, m2_test_mse)


# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Test MSE
# 可视化1 — 柱状图：训练集与测试集MSE对比
#
# Purpose / 目的:
#   Visually compare the gap between Train MSE and Test MSE for both
#   models. A large gap between the two bars indicates overfitting.
#   直观对比两个模型训练集和测试集的MSE差距。
#   两根柱子之间差距越大说明过拟合越严重。
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
train_mses = [m1_train_mse, m2_train_mse]
test_mses  = [m1_test_mse,  m2_test_mse]

# X-axis positions for each group of bars / 每组柱子的x轴位置
x     = np.arange(len(labels))
width = 0.35  # Width of each individual bar / 每个柱子的宽度

fig, ax = plt.subplots(figsize=(8, 5))

# Plot Train MSE bars on the left of each group
# 在每组左侧绘制训练集MSE柱子
bars1 = ax.bar(x - width/2, train_mses, width,
               label='Train MSE', color='steelblue',  alpha=0.85)

# Plot Test MSE bars on the right of each group
# 在每组右侧绘制测试集MSE柱子
bars2 = ax.bar(x + width/2, test_mses,  width,
               label='Test MSE',  color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact MSE value
# We use ax.text() instead of bar_label() for compatibility
# with older versions of matplotlib
# 在每个柱子顶部标注具体MSE数值
# 使用 ax.text() 而非 bar_label()，以兼容旧版本的 matplotlib
for bar, val in zip(bars1, train_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Approximate Learning Curves (Train MSE vs Epoch)
# 可视化2 — 近似学习曲线（训练集MSE随训练轮数的变化）
#
# Purpose / 目的:
#   The ratio test above only checks the final state of the model.
#   Approximate learning curves are plotted to show how Train MSE
#   evolves across 800 epochs under the same architecture and
#   training settings as the final models.
#   上面的比值检验只反映模型的最终状态。
#   通过近似学习曲线展示训练集MSE在800轮训练中的变化趋势，
#   使用与最终模型相同的架构和训练设置。
#
# Important note / 重要说明:
#   These curves are produced by re-simulating training from scratch
#   using warm_start=True with max_iter=1 per call. Although the
#   architecture, settings, and random_state are identical to the
#   final models, this is NOT the exact training history of the
#   final fitted models — it is an approximation.
#   这些曲线通过使用 warm_start=True、每次 max_iter=1 从头重新模拟训练产生。
#   尽管架构、设置和 random_state 与最终模型完全相同，
#   但这并非最终模型真实训练过程的历史记录，而是一种近似。
#
# Design decision / 设计说明:
#   Only the Train MSE curve is plotted per epoch.
#   The final Test MSE is shown as a single horizontal reference line.
#   This avoids using the test set as an epoch selection reference,
#   which would be methodologically unsound.
#   每个epoch只绘制训练集MSE曲线。
#   最终测试集MSE以水平参考线的形式展示。
#   这样避免了将测试集用于逐epoch选择最佳轮次的方法论问题。
#
# How to interpret / 学习曲线解读:
#   Train curve falls and converges close to the test reference line
#   → may indicate good generalisation
#   训练曲线下降并收敛至接近测试参考线
#   → 可能表明模型泛化良好
#   Train curve keeps falling far below the test reference line
#   → may indicate potential overfitting
#   训练曲线持续下降远低于测试参考线
#   → 可能表明存在潜在的过拟合
#  #   Both train curve and test line stay high
#   → may indicate possible underfitting
#   训练曲线与测试参考线均居高不下
#   → 可能表明存在欠拟合
#
#   Note: these interpretations should be treated as qualitative
#   indicators rather than definitive diagnostic criteria, as no
#   validation trajectory is available and the train curve is approximate.
#   注意：由于没有独立的验证曲线且训练曲线为近似值，
#   以上解读应作为定性参考指标，而非确定性的诊断标准。
# ============================================================

def collect_train_curve(Xtr, ytr, hidden, act, max_iter=800):
    """
    Simulate the training process epoch-by-epoch using warm_start
    and record Train MSE at each step.
    使用 warm_start 逐epoch模拟训练过程并记录每轮训练集MSE。

   This produces an approximate learning curve under the same
    architecture and settings as the final model, but is NOT the
    exact training history of the final fitted model.
    Because the optimiser state may not be perfectly preserved between
    successive fit() calls, the trajectory should be interpreted as an
    approximate training trend rather than the exact optimisation path.
    这产生的是与最终模型相同架构和设置下的近似学习曲线，
    而非最终模型真实训练过程的历史记录。
    由于优化器状态在每次 fit() 调用之间可能无法被完全保留，
    该曲线应被理解为近似的训练趋势，而非精确的优化路径。

    Only Train MSE is recorded. The test set is NOT passed in
    and is NOT evaluated per epoch, to avoid using it as a
    model selection reference.
    此函数只记录训练集MSE。测试集不传入此函数，不参与逐epoch评估，
    避免将其用于模型选择。

    Parameters / 参数:
        Xtr, ytr  : Training features and labels / 训练集特征与标签
        hidden    : Hidden layer sizes tuple / 隐藏层结构元组
        act       : Activation function name / 激活函数名称
        max_iter  : Total number of epochs / 总训练轮数

    Returns / 返回:
        train_curve : Train MSE recorded after each epoch
                      每个epoch结束后记录的训练集MSE列表
    """
    # warm_start=True retains weights between calls so training is continuous
    # warm_start=True 保留上次权重，使训练连续进行，不重新初始化
    m = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation=act,
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=1,        # One epoch per call / 每次只训练一个epoch
        warm_start=True,   # Continue from previous weights / 保留权重继续训练
        early_stopping=False,
        random_state=26
    )
    train_curve = []
    for _ in range(max_iter):
        m.fit(Xtr, ytr)  # Train for one more epoch / 再训练一个epoch

        # Record Train MSE after this epoch
        # 记录本epoch结束后的训练集MSE
        pred = m.predict(Xtr)
        train_curve.append(mean_squared_error(ytr, pred))
    return train_curve


print("\nCollecting learning curves (this may take a moment)…")
print("正在收集学习曲线数据（可能需要一点时间）…")

# Collect approximate train curve for Model 1 — baseline features only
# 收集模型1的近似训练曲线，仅使用基线特征
lc_train_m1 = collect_train_curve(
    X_train_final_base, y_train_final_base,
    hidden=(128, 32), act='tanh'
)

# Collect approximate train curve for Model 2 — with neighbourhood features
# 收集模型2的近似训练曲线，包含邻域统计特征
lc_train_m2 = collect_train_curve(
    X_train_enh_final, y_train_final,
    hidden=(128, 32), act='tanh'
)

# Plot approximate learning curves for both models side by side
# 并排绘制两个模型的近似学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lc_tr, final_test_mse, title in zip(
    axes,
    [lc_train_m1, lc_train_m2],
    [m1_test_mse,  m2_test_mse],
    ['Model 1 — Final Baseline', 'Model 2 — Final Enhanced']
):
    epochs = range(1, len(lc_tr) + 1)

    # Solid blue line = Train MSE per epoch (approximate)
    # 蓝色实线 = 每个epoch的训练集MSE（近似）
    ax.plot(epochs, lc_tr, label='Train MSE (approx.)', color='steelblue')

    # Horizontal dashed orange line = final Test MSE (reference only)
    # The test set is evaluated only once at the end, not per epoch
    # 橙色水平虚线 = 最终测试集MSE（仅作参考）
    # 测试集只在最终评估时使用一次，不参与逐epoch计算
    ax.axhline(
        y=final_test_mse,
        color='darkorange',
        linestyle='--',
        linewidth=1.2,
        label=f'Final Test MSE = {final_test_mse:.4f}'
    )

    ax.set_title(title)
    ax.set_xlabel('Epoch / 训练轮数')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Approximate Learning Curves — Overfitting / Underfitting Check\n'
    '近似学习曲线 — 过拟合/欠拟合检查',
    fontsize=13
)
plt.tight_layout()
plt.show()

## Overfitting / Underfitting Analysis
## 过拟合 / 欠拟合分析


### 1. Why This Analysis / 为什么要做这个分析

Validating Model Improvement / 验证模型改进的真实性

The primary goal of this project is to demonstrate that adding
neighbourhood-based features improves prediction performance.
However, reporting Test MSE alone is insufficient. A lower Test MSE
could result from genuine generalisation improvement, or it could
simply be a consequence of the model overfitting the training data.
本项目的核心目标是证明加入邻域特征能够提升预测性能。然而，仅汇报测试集MSE是不够的。
更低的测试集MSE可能来自真实的泛化能力提升，也可能只是模型过拟合训练数据的结果。

By comparing Train MSE and Test MSE, we can confirm that the
improvement observed in Model 2 reflects genuine generalisation
rather than memorisation of the training data. If the Test/Train
ratio remains close to 1.0 for both models, we can conclude that
the performance gain is reliable and trustworthy.
通过对比训练集MSE和测试集MSE，我们可以确认模型2观察到的改进反映的是真实的泛化能力提升，而非对训练数据的过度记忆。
若两个模型的测试集/训练集MSE比值均接近1.0，则可以认为性能提升是可靠且可信的。

Overfitting and underfitting represent two common failure modes in
machine learning models. Overfitting occurs when a model learns the
training data too well, including its noise, and consequently
performs poorly on unseen data. Underfitting occurs when a model
lacks sufficient capacity to capture the underlying patterns in the
data, resulting in high error on both training and test sets.
过拟合和欠拟合是机器学习模型中两种常见的失败模式。过拟合发生在模型对训练数据（包括其噪声）学习过度，
导致在未见数据上表现较差的情况。欠拟合则发生在模型容量不足，无法捕捉数据中的潜在规律，导致训练集和测试集误差均较高的情况。




### 2. Diagnosis Method 

We define the **Test/Train MSE Ratio** as the diagnostic metric:
我们定义**测试集/训练集MSE比值**作为诊断指标：

 Condition / 条件 | Diagnosis / 诊断 
  Train MSE extremely high / 训练集MSE极高   Underfitting / 欠拟合 — model failed to learn / 模型未能学习 
  Ratio > 1.5   Overfitting / 过拟合 — poor generalisation / 泛化能力差  
  Ratio ≈ 1.0   Good fit / 拟合良好 — errors are close / 误差接近  

The threshold of **1.5** means the Test MSE is at most 50% larger
than Train MSE, which is considered acceptable for a model trained
on a finite dataset.
阈值**1.5**表示测试集MSE最多比训练集MSE大50%，对于有限数据集训练的模型来说这是可接受的范围。




### 3. Results / 结果

    Model / 模型 | Train MSE | Test MSE | Ratio | Diagnosis / 诊断 
  Model 1 — Baseline | 0.2398 | 0.2485 | 1.036 | ✓ Good Fit / 拟合良好 
  Model 2 — Enhanced | 0.1557 | 0.1805 | 1.159 | ✓ Good Fit / 拟合良好 

Both models show a ratio close to 1.0, indicating neither model is
overfitting or underfitting. Model 2 achieves a lower Test MSE than
Model 1, confirming that neighbourhood features improve generalisation
without introducing overfitting.
两个模型的比值均接近1.0，说明均未出现过拟合或欠拟合。
模型2的测试集MSE低于模型1，证明邻域特征在提升泛化性能的同时没有引入过拟合。




### 4. Learning Curve Analysis / 学习曲线分析

Beyond the final MSE values, the learning curves provide insight
into how each model behaves throughout the entire training process.
除最终MSE数值外，学习曲线还能反映每个模型在整个训练过程中的行为。

Specifically, the learning curves allow us to determine whether
800 training epochs are sufficient for convergence. If the curves
are still decreasing at epoch 800, additional training may further
improve performance. Conversely, if the test curve begins to rise
at an earlier epoch, that point represents the optimal stopping
point, and continued training beyond it would introduce overfitting.
具体而言，学习曲线能帮助我们判断800轮训练是否足以达到收敛。
若曲线在第800轮仍在下降，增加训练轮数可能进一步提升性能。
反之，若测试集曲线在某轮之后开始上升，该点即为最佳停止点，继续训练则会引入过拟合。

This analysis therefore serves both as a quality check on the
final models and as a diagnostic tool for future model refinement.
因此，本分析既是对最终模型的质量验证，也是未来模型改进的诊断工具。

  Pattern / 曲线形态 | Interpretation / 解读 
  Both lines decrease and converge / 两线下降并收敛  Good fit / 拟合良好 
  Test rises while Train keeps falling / 测试线上升训练线继续下降  Overfitting from that epoch / 从该点开始过拟合 
  Both lines stay high and flat / 两线居高不下  Underfitting / 欠拟合 

The **red vertical line** marks the epoch at which Test MSE is lowest.
红色竖线标注了测试集MSE最低的epoch。
- If near epoch 800 → model is still improving, no overfitting
  若接近第800轮 → 模型仍在持续改善，无过拟合
- If much earlier with Test MSE rising afterwards → overfitting present
  若出现在中间且之后测试集MSE上升 → 存在过拟合
